# PO fine-tuning and local adapter merge

The original GPU training and evaluation cells are commented out for now.
`MODE = "merge"` runs the local merge.
Run this notebook from the project root or `backend/`. It uses the existing local
adapter and the already cached original 4-bit base weights. The merge cell uses
`local_files_only=True`, so it does not download weights again.
No Google Drive, Colab, CUDA, or Unsloth installation is required for CPU merging.

The CPU path uses Transformers + PEFT + bitsandbytes and the original
`unsloth/gemma-4-E4B-it-unsloth-bnb-4bit` checkpoint. PEFT dequantizes each adapted
linear layer, adds the LoRA update, and requantizes it. The saved output remains
bitsandbytes-quantized. Requantization can change outputs slightly; evaluate it.
The model remains multimodal and retains the saved processor and chat template.

Install merge dependencies in this notebook's local Python environment if needed,
then restart the kernel before running the notebook. The versions below match the
reported training environment and adapter metadata. The cached base and
merged output each need substantial disk space. CPU merging can require more RAM
than is currently free on this machine; close memory-heavy applications first.
A kernel killed during loading/merging usually requires more available RAM.

For optional retraining, uncomment the training cells, set `MODE = "train"`,
install compatible Unsloth/TRL/GPU dependencies separately, and use a CUDA GPU. Training writes to a separate local
run directory, preserving the downloaded final adapter.


In [ ]:
import json
from pathlib import Path

MODE = "merge"  # Training cells are commented out; keep "merge" for local merging.
if MODE not in {"merge", "train"}:
    raise ValueError('MODE must be "merge" or "train"')

# Works when the notebook kernel starts at the project root or in backend/.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "backend" / "po_classification.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the kernel in the po_classification project or backend folder")

ADAPTER_DIR = PROJECT_ROOT / "final_adapter-20261126T064032Z-1-001" / "final_adapter"
MERGED_DIR = PROJECT_ROOT / "po_gemma4_merged"
DATASET_DIR = PROJECT_ROOT / "output" / "po_dataset"
TRAIN_OUTPUT_DIR = PROJECT_ROOT / "output" / "gemma4_po_run"

for filename in ("adapter_config.json", "adapter_model.safetensors"):
    if not (ADAPTER_DIR / filename).is_file():
        raise FileNotFoundError(ADAPTER_DIR / filename)

adapter_config = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
recorded_base = adapter_config["base_model_name_or_path"]
# Keep the exact original 4-bit base recorded in the adapter configuration.
CPU_BASE_MODELS = {
    "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit": "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
}
if recorded_base not in CPU_BASE_MODELS:
    raise ValueError(f"Verify the supported 4-bit base for {recorded_base!r}")
BASE_MODEL_ID = CPU_BASE_MODELS[recorded_base]

print("Mode:", MODE)
print("Adapter:", ADAPTER_DIR)
print("Recorded training base:", recorded_base)
print("CPU merge base (cached weights):", BASE_MODEL_ID)
print("Merged output:", MERGED_DIR)


In [ ]:
# Uncomment and run once if these packages are not installed in this kernel.
# Restart the kernel after installation, then run the notebook again.

# %pip install torch "transformers==5.5.0" "peft==0.21.2" "bitsandbytes==0.50.2" accelerate safetensors pillow sentencepiece torchvision


## Optional GPU training (commented out)

Training and evaluation code is preserved as comments. Uncomment it and set
`MODE = "train"` only when intentionally retraining.


In [ ]:
# if MODE == "train":
#     root = DATASET_DIR
#     assert root.is_dir(), f"Dataset directory does not exist: {root}"


In [ ]:
# if MODE == "train":
#     import sys
#     sys.path.insert(0, str(root))

In [ ]:
# if MODE == "train":
#     from load_dataset import load_examples
#     sample = next(load_examples(root, "train"))
#
#     display(sample["images"][0])
#     print(sample["messages"][-1]["content"][0]["text"])

In [ ]:
# if MODE == "train":
#     import unsloth
#     from unsloth import FastVisionModel
#     import torch
#
#     assert torch.cuda.is_available(), "Training requires a local CUDA GPU; use MODE = 'merge' for CPU merging"
#
#     model, processor = FastVisionModel.from_pretrained(
#         model_name="unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
#         load_in_4bit=True,
#         use_gradient_checkpointing="unsloth",
#     )
#
#     print("Gemma 4 E4B loaded successfully")

In [ ]:
# if MODE == "train":
#     from unsloth import FastVisionModel, get_chat_template
#
#     processor = get_chat_template(processor, "gemma-4")
#
#     model = FastVisionModel.get_peft_model(
#         model,
#         finetune_vision_layers=False,
#         finetune_language_layers=True,
#         finetune_attention_modules=True,
#         finetune_mlp_modules=True,
#         r=8,
#         lora_alpha=8,
#         lora_dropout=0,
#         bias="none",
#         random_state=42,
#     )
#
#     FastVisionModel.for_training(model)
#     model.print_trainable_parameters()

In [ ]:
# if MODE == "train":
#     import json
#     from pathlib import Path
#     from PIL import Image
#     from torch.utils.data import Dataset
#
#     root = DATASET_DIR
#     assert root.exists(), "Update root to your extracted dataset folder"
#
#     instructions = (root / "instructions.txt").read_text()
#
#     class PODataset(Dataset):
#         def __init__(self, split):
#             self.rows = [
#                 json.loads(line)
#                 for line in (root / f"{split}.jsonl").read_text().splitlines()
#                 if line.strip()
#             ]
#
#         def __len__(self):
#             return len(self.rows)
#
#         def __getitem__(self, index):
#             row = self.rows[index]
#             content = []
#
#             for filename in row["images"]:
#                 with Image.open(root / filename) as image:
#                     content.append({
#                         "type": "image",
#                         "image": image.convert("RGB"),
#                     })
#
#             content.append({"type": "text", "text": row["prompt"]})
#
#             return {"messages": [
#                 {
#                     "role": "system",
#                     "content": [{"type": "text", "text": instructions}],
#                 },
#                 {"role": "user", "content": content},
#                 {
#                     "role": "assistant",
#                     "content": [{
#                         "type": "text",
#                         "text": json.dumps(row["target"], ensure_ascii=False),
#                     }],
#                 },
#             ]}
#
#     train_data = PODataset("train")
#     validation_data = PODataset("validation")
#
#     print("Training documents:", len(train_data))
#     print("Validation documents:", len(validation_data))

In [ ]:
# if MODE == "train":
#     from unsloth.trainer import UnslothVisionDataCollator
#
#     collator = UnslothVisionDataCollator(
#         model,
#         processor,
#         resize="max",
#         train_on_responses_only=True,
#         instruction_part="<|turn>user\n",
#         response_part="<|turn>model\n",
#     )
#
#     # Preserve full instructions, page tokens and JSON answers.
#     collator.max_seq_length = None
#     collator.truncation = False
#
#     batch = collator([train_data[0]])
#
#     print("Input shape:", batch["input_ids"].shape)
#     print("Answer tokens:", (batch["labels"] != -100).sum().item())
#
#     assert (batch["labels"] != -100).any(), "No answer tokens found"
#     del batch

In [ ]:
# if MODE == "train":
#     from trl import SFTTrainer, SFTConfig
#     from transformers.trainer_utils import get_last_checkpoint
#
#     output_dir = str(TRAIN_OUTPUT_DIR)
#
#     trainer = SFTTrainer(
#         model=model,
#         processing_class=processor.tokenizer,
#         data_collator=collator,
#         train_dataset=train_data,
#         eval_dataset=validation_data,
#         args=SFTConfig(
#             output_dir=output_dir,
#             per_device_train_batch_size=1,
#             per_device_eval_batch_size=1,
#             gradient_accumulation_steps=4,
#             num_train_epochs=1,
#             learning_rate=1e-4,
#             warmup_steps=2,
#             max_grad_norm=0.3,
#             fp16=True,
#             bf16=False,
#             optim="adamw_8bit",
#             logging_steps=1,
#             eval_strategy="epoch",
#             save_strategy="steps",
#             save_steps=5,
#             save_total_limit=2,
#             remove_unused_columns=False,
#             dataset_text_field="",
#             dataset_kwargs={"skip_prepare_dataset": True},
#             max_length=None,
#             report_to="none",
#             dataloader_num_workers=0,
#             seed=42,
#         ),
#     )
#
#     checkpoint = (
#         get_last_checkpoint(output_dir)
#         if Path(output_dir).exists() else None
#     )
#
#     trainer.train(resume_from_checkpoint=checkpoint)
#
#     trainer.save_model(f"{output_dir}/final_adapter")
#     processor.save_pretrained(f"{output_dir}/final_adapter")

In [ ]:
# if MODE == "train":
#     import torch
#     from unsloth import FastVisionModel
#
#     FastVisionModel.for_inference(model)
#
#     test_data = PODataset("test")
#     example = test_data[12]
#
#     # Exclude the expected answer from the model input.
#     messages = example["messages"][:-1]
#
#     images = [
#         part["image"]
#         for message in messages
#         for part in message["content"]
#         if part["type"] == "image"
#     ]
#
#     text = processor.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#         enable_thinking=False,
#     )
#
#     inputs = processor(
#         text=text,
#         images=images,
#         add_special_tokens=False,
#         return_tensors="pt",
#     ).to("cuda")
#
#     with torch.inference_mode():
#         output = model.generate(
#             **inputs,
#             max_new_tokens=4096,
#             do_sample=False,
#             use_cache=True,
#         )
#
#     prediction = processor.tokenizer.decode(
#         output[0, inputs["input_ids"].shape[1]:],
#         skip_special_tokens=True,
#     )
#
#     print("MODEL OUTPUT:\n", prediction)
#     print("\nEXPECTED JSON:\n",
#           example["messages"][-1]["content"][0]["text"])

## Merge the existing adapter into the cached 4-bit base on CPU

The merge cell reads the original bitsandbytes checkpoint from the local Hugging
Face cache, attaches the saved adapter, and merges one adapted layer at a time.
A warning about rounding differences during 4-bit merging is expected.
The adapter directory is read-only input. Existing nonempty output directories
are rejected to avoid mixing checkpoints; choose a fresh `MERGED_DIR` to retry.

The output is a bitsandbytes-quantized Hugging Face model, **not yet an
Ollama/GGUF model**. A compatible conversion preserving vision components is
still required. Restart the kernel after a failed model load to free memory.
If an editor reports that this notebook changed on disk, reload the saved file
before executing to avoid restoring outdated cells.


In [ ]:
if MODE == "merge":
    import gc
    import shutil

    import torch
    from peft import PeftModel
    from transformers import AutoProcessor, Gemma4ForConditionalGeneration

    if MERGED_DIR.exists() and (not MERGED_DIR.is_dir() or any(MERGED_DIR.iterdir())):
        raise FileExistsError(f"Choose a fresh MERGED_DIR; output already exists: {MERGED_DIR}")

    print("Free disk space (GiB):", round(shutil.disk_usage(PROJECT_ROOT).free / 2**30, 1))
    print("Loading processor from the local adapter...")
    processor = AutoProcessor.from_pretrained(str(ADAPTER_DIR), local_files_only=True)

    print("Loading cached 4-bit base on CPU:", BASE_MODEL_ID)
    base_model = Gemma4ForConditionalGeneration.from_pretrained(
        BASE_MODEL_ID,
        dtype=torch.bfloat16,
        device_map="cpu",
        low_cpu_mem_usage=True,
        local_files_only=True,
    )
    if not getattr(base_model, "is_loaded_in_4bit", False):
        raise RuntimeError("Expected the recorded bitsandbytes 4-bit base to load in 4-bit")

    model = PeftModel.from_pretrained(
        base_model,
        str(ADAPTER_DIR),
        is_trainable=False,
        autocast_adapter_dtype=False,
        low_cpu_mem_usage=True,
    )
    print("Merging local LoRA weights into the 4-bit base...")
    with torch.no_grad():
        merged_model = model.merge_and_unload(safe_merge=True)

    merged_model.save_pretrained(
        str(MERGED_DIR),
        safe_serialization=True,
        # Transformers 5.5 cannot reverse Bnb4bitDeserialize when saving.
        save_original_format=False,
        max_shard_size="2GB",
    )
    processor.save_pretrained(str(MERGED_DIR))

    # Keep the exact training template, including any custom Unsloth edits.
    template = ADAPTER_DIR / "chat_template.jinja"
    if template.is_file():
        shutil.copy2(template, MERGED_DIR / template.name)

    del merged_model, model, base_model
    gc.collect()
    print("Merged 4-bit Hugging Face model and processor saved to:", MERGED_DIR)
